# Core Validation 007 — Resumable Confirmation v1.1

Discovery is already frozen and the winner is `interference_cut`. The original 80711/80712/80713 confirmation set is retired after the monolithic runner exposed 80711 and terminated during 80712 before a seed checkpoint existed. This notebook runs the amended untouched confirmation seeds 80721/80722/80723 only.

Enable Internet and a GPU accelerator. The confirmation orchestrator runs one seed per fresh Python/CUDA process, checkpoints and pushes after every seed, and resumes from canonical partial artifacts after a Kaggle session restart.


In [ ]:
import os, subprocess, sys
REPO='/kaggle/working/mini-cells'
BRANCH='codex/core-validation-007-functional-boundary-discovery'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/ArcheLabs/mini-cells.git',REPO],check=True)
os.chdir(REPO)
subprocess.run(['git','fetch','origin',BRANCH],check=True)
subprocess.run(['git','checkout',BRANCH],check=True)
subprocess.run(['git','merge','--ff-only',f'origin/{BRANCH}'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm]'],check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
github_token=None
github_source=None
for name in ('GITHUB_TOKEN','GH_TOKEN'):
    try:
        value=secrets.get_secret(name)
    except Exception:
        value=None
    if value and github_token is None:
        github_token=value
        github_source=name
assert github_token, 'Add Kaggle Secret GITHUB_TOKEN or GH_TOKEN with Contents read/write before running confirmation.'
os.environ['GITHUB_TOKEN']=github_token
os.environ['GH_TOKEN']=github_token
try:
    hf_token=secrets.get_secret('HF_TOKEN')
except Exception:
    hf_token=None
if hf_token:
    os.environ['HF_TOKEN']=hf_token
print('GitHub credential loaded from:', github_source, '; HF_TOKEN:', bool(hf_token))


In [ ]:
import torch
assert torch.cuda.is_available(), 'Core 007 confirmation requires CUDA'
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'scripts/research/publish_core_validation_007.py','--preflight-only','--branch',BRANCH,'--secret-name','GITHUB_TOKEN'],check=True)


## Run / resume amended confirmation

This cell is idempotent. Re-run the same cell after an interruption. Completed matching seed checkpoints are hydrated from the repository and skipped; only pending/failed seeds run again. Every completed seed is reported, committed, and pushed before the next seed starts.


In [ ]:
subprocess.run([sys.executable,'scripts/research/orchestrate_core_validation_007_confirmation.py','--branch',BRANCH,'--secret-name','GITHUB_TOKEN','--push-results'],check=True)


In [ ]:
from pathlib import Path
decision=Path('results/core-validation-007-functional-boundary-discovery/confirmation/decision.json')
gates=Path('results/core-validation-007-functional-boundary-discovery/confirmation/gate-summary.csv')
print(decision.read_text())
print(gates.read_text() if gates.exists() else 'gate-summary.csv not written yet')
